# Ch.1 データ読み込み・集計（pandas）

## 本チャプターのゴール
- Pythonでデータを読み込み、基本的な情報を確認できる
- 欠損値の確認・処理ができる
- グループ集計・フィルタリングができる

## 使用データ
scikit-learnに内蔵されているワインデータセット（`load_wine`）
- 178件のワインデータ（イタリア産の3品種）
- アルコール度数・色の濃さ・プロリン含有量 など13種類の化学成分
- ターゲット変数：ワインの品種（0 / 1 / 2）

---

## 🤖 生成AIの使い方ガイド（本チャプター）

| AIに任せてよいこと | 自分で理解すること |
|---|---|
| `read_csv` のオプション（encoding など） | どの列を分析するか、なぜその集計が必要か |
| エラーメッセージの修正 | 集計結果の数値が何を意味するか |
| `groupby` の基本的な書き方 | 欠損値をどう処理するかの判断 |

**プロンプトテンプレート**
```
【目的】ワインデータの品種ごとにアルコール度数の平均を集計したい
【使うライブラリ】pandas
【制約】Python 3.8、インターネット接続なし
【わからない点】groupby の書き方
```

---
## 1.1 データの読み込みと基本確認

In [ ]:
import pandas as pd
import numpy as np
from sklearn.datasets import load_wine

# データの読み込み
wine = load_wine()

# DataFrameに変換
df = pd.DataFrame(wine.data, columns=wine.feature_names)

# 品種ラベルを追加
df['target'] = wine.target
df['品種'] = df['target'].map({0: 'Barolo', 1: 'Grignolino', 2: 'Barbera'})

print('データ件数:', len(df))
print('列数:', len(df.columns))

In [ ]:
# 先頭5行を確認
df.head()

In [ ]:
# データ型と件数の確認
df.info()

In [ ]:
# 基本統計量の確認
df.describe()

---
## 1.2 欠損値の確認と処理

実務のデータには欠損値（NaN）が必ず含まれます。
ここでは演習のために、意図的に欠損値を追加して処理を練習します。

In [ ]:
# 演習用：欠損値を人工的に追加
df_with_na = df.copy()
np.random.seed(42)
for col in ['alcohol', 'color_intensity', 'proline']:
    idx = np.random.choice(df_with_na.index, size=10, replace=False)
    df_with_na.loc[idx, col] = np.nan

# 欠損値の確認
print('各列の欠損値件数：')
print(df_with_na.isnull().sum())

In [ ]:
# 欠損値の割合を確認
missing_rate = df_with_na.isnull().sum() / len(df_with_na) * 100
print('欠損率（%）：')
print(missing_rate[missing_rate > 0].round(1))

In [ ]:
# 方法1：欠損値を平均値で補完
df_filled = df_with_na.copy()
for col in ['alcohol', 'color_intensity', 'proline']:
    mean_val = df_filled[col].mean()
    df_filled[col] = df_filled[col].fillna(mean_val)
    print(f'{col}: 平均値 {mean_val:.2f} で補完')

print('\n補完後の欠損値件数：', df_filled.isnull().sum().sum())

In [ ]:
# 方法2：欠損値のある行を削除
df_dropped = df_with_na.dropna()
print('削除前:', len(df_with_na), '件')
print('削除後:', len(df_dropped), '件')

# ポイント：データ量が多く欠損が少ない場合は削除も選択肢。
# 欠損が多い場合は補完が現実的。

---
## 1.3 フィルタリングと集計

In [ ]:
# 品種ごとの件数を確認
print(df['品種'].value_counts())

In [ ]:
# フィルタリング：アルコール度数が13以上のワイン
high_alcohol = df[df['alcohol'] >= 13.0]
print(f'アルコール度数13以上: {len(high_alcohol)} 件')
high_alcohol.head()

In [ ]:
# groupby：品種ごとの平均値
group_mean = df.groupby('品種')[['alcohol', 'color_intensity', 'proline']].mean().round(2)
print('品種ごとの平均値：')
group_mean

In [ ]:
# ソート：色の濃さが高い順に並べる
df_sorted = df.sort_values('color_intensity', ascending=False)
df_sorted[['品種', 'alcohol', 'color_intensity']].head(10)

---
## 🖊️ 演習（40分）

以下の問いに答えるコードを書いてください。

**ヒント：** まずAIに叩き台コードを生成させ、実行して結果を確認してから「何がわかるか」を自分の言葉でまとめましょう。

---

### 問1：品種ごとのデータ件数を確認する

In [ ]:
# TODO: df を使って、品種（'品種'列）ごとのデータ件数を表示してください
# ヒント：value_counts() を使います



### 問2：プロリン含有量が最も多い品種を特定する

In [ ]:
# TODO: 品種ごとのプロリン（'proline'）の平均値を集計し、
#       最も平均値が高い品種を表示してください
# ヒント：groupby → mean → sort_values または idxmax を使います



### 問3：アルコール度数が高く、色が濃いワインを絞り込む

In [ ]:
# TODO: 以下の条件を両方満たすデータを抽出してください
#   - alcohol >= 13.5
#   - color_intensity >= 5.0
# 何件ありましたか？品種の内訳は？



### 問4（発展）：欠損値補完の方法を品種ごとの平均に変更する

In [ ]:
# TODO: df_with_na のアルコール度数（'alcohol'）の欠損値を、
#       「全体の平均」ではなく「品種ごとの平均」で補完してください
# ヒント：groupby と transform を組み合わせます（AIに聞いてみましょう）



---
## ✅ チャプターのまとめ

| 操作 | コード |
|------|--------|
| データ読み込み | `pd.DataFrame(data, columns=columns)` |
| 基本確認 | `head()`, `info()`, `describe()` |
| 欠損値確認 | `isnull().sum()` |
| 欠損値補完 | `fillna(value)` |
| フィルタリング | `df[条件式]` |
| グループ集計 | `groupby('列名').mean()` |
| ソート | `sort_values('列名', ascending=False)` |